# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AshenDary/Week1_RunTheStarterNotebooks/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook defines the data contract for the FlyRank Search Intelligence / Content Refresh Priority Scoring lane. It uses the gated Hugging Face warehouse release and the mid-panel development partition `month = '2026-03'`. The final month, `2026-06`, is treated as sealed test data and is not queried here.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn pandas numpy

import os

import duckdb
import numpy as np
import pandas as pd

try:
    from google.colab import userdata
except ImportError:
    userdata = None


DEV_MONTH = "2026-03"
SEALED_TEST_MONTH = "2026-06"
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_DAILY_MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month={DEV_MONTH}/*.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN and userdata is not None:
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

con = duckdb.connect()
EXECUTE_REMOTE_QUERIES = bool(HF_TOKEN)
if EXECUTE_REMOTE_QUERIES:
    con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [HF_TOKEN])


def run_sql(query: str, label: str) -> pd.DataFrame:
    print(f"\n--- {label} ---")
    print(query.strip())
    if not EXECUTE_REMOTE_QUERIES:
        print("\nSkipped remote execution: set HF_TOKEN in the environment or Colab Secrets to run this query.")
        return pd.DataFrame()
    result = con.sql(query).df()
    display(result)
    return result


print(f"Development month: {DEV_MONTH}")
print(f"Sealed test month not used: {SEALED_TEST_MONTH}")
print(f"Remote query execution enabled: {EXECUTE_REMOTE_QUERIES}")


Development month: 2026-03
Sealed test month not used: 2026-06
Remote query execution enabled: True


## 1. Unit of Analysis + Time Window

One row in the raw warehouse daily fact slice is one **content page on one report date for one pseudonymized client**.

The development time window is `month = '2026-03'`, which means March 1, 2026 through March 31, 2026. The final month (`2026-06`) is sealed test data and is not used in this notebook.

The grain is uniquely identified by:

- `report_date`
- `client_hash_id`
- `content_hash_id`

This contract starts from the daily fact table because the Search Intelligence / refresh lane needs time-aware features, lagged behavior, and future-window labels without peeking into sealed test data.

In [2]:
section_1_sample_query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM {FACT_DAILY_MARCH}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
ORDER BY report_date, client_hash_id, content_hash_id
LIMIT 3
"""

section_1_sample = run_sql(section_1_sample_query, "Section 1: three March rows at the claimed grain")



--- Section 1: three March rows at the claimed grain ---
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
ORDER BY report_date, client_hash_id, content_hash_id
LIMIT 3


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position
0,2026-03-01,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,0,0,NaN
1,2026-03-01,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0,0,NaN
2,2026-03-01,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0,0,NaN


## 2. Fields: Feature / Label / Context / Excluded

This contract uses a small, explicit feature set so each field can be defended as knowable at the decision moment. Runtime code below also inspects the actual warehouse schema and categorizes every discovered column.

| Bucket | Field | Source | Use / Reason |
|---|---|---|---|
| Feature | `decision_dow` | derived from `report_date` | Day of week for the decision date; known before ranking. |
| Feature | `gsc_impressions_lag1` | derived from `gsc_impressions` | Prior-day search demand signal. |
| Feature | `gsc_clicks_lag1` | derived from `gsc_clicks` | Prior-day click capture signal. |
| Feature | `gsc_ctr_7d_past` | derived from past `gsc_clicks / gsc_impressions` | Seven-day trailing CTR, excluding the decision day and future days. |
| Feature | `gsc_position_7d_past_avg` | derived from past `gsc_avg_position` | Seven-day trailing average rank position, excluding the decision day and future days. |
| Label / proxy | `label_next_day_gsc_clicks` | derived from next-day `gsc_clicks` inside March | Regression proxy for near-term search value after the decision moment. |
| Context | `report_date` | fact table | Defines time and supports splits/windows; not a model feature except derived `decision_dow`. |
| Context | `client_hash_id` | fact/dim tables | Join/group/split key only; never a model feature. |
| Context | `content_hash_id` | fact/dim tables | Unit identifier and join key only; never a model feature. |
| Context | `ga4_data_available`, `gsc_data_available` | fact table, if present | Availability filters/checks only; not model features here. |
| Context | `gsc_data_start`, `ga4_data_start`, `access_profile` | `dim_clients` | Coverage and limitation checks only. |
| Context | `url_hash_id`, `keyword_hash_id`, content metadata | `dim_content`, if joined later | Grouping/case-analysis context only unless separately reviewed. |
| Excluded | current-day `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` | fact table | Not used directly because same-day values may not be known at the ranking moment. |
| Excluded | future values such as `label_next_day_gsc_clicks` | derived label | Target information; using it as a feature would leak the answer. |
| Excluded | final-month rows (`month = '2026-06'`) | fact table | Sealed test period; not used for development. |
| Excluded | raw/scrambled IDs as model inputs | fact/dim tables | Identifiers are for joins and grouping, not signal. |
| Excluded | GA4 engagement metrics for this contract | fact table | Availability is sparse and three-valued; they need a separate availability-aware contract before modeling. |
| Excluded | unknown extra columns discovered at runtime | schema inspection | Excluded until reviewed and assigned to a contract bucket. |

In [3]:
feature_fields = [
    "decision_dow",
    "gsc_impressions_lag1",
    "gsc_clicks_lag1",
    "gsc_ctr_7d_past",
    "gsc_position_7d_past_avg",
]
label_fields = ["label_next_day_gsc_clicks"]
context_fields = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "ga4_data_available",
    "gsc_data_available",
    "gsc_data_start",
    "ga4_data_start",
    "access_profile",
    "url_hash_id",
    "keyword_hash_id",
]
feature_source_fields = ["gsc_impressions", "gsc_clicks", "gsc_avg_position"]


def describe_relation(relation_sql: str, source_name: str) -> pd.DataFrame:
    if not EXECUTE_REMOTE_QUERIES:
        return pd.DataFrame(columns=["source", "column_name", "column_type"])
    schema = con.sql(f"DESCRIBE SELECT * FROM {relation_sql} LIMIT 0").df()
    schema.insert(0, "source", source_name)
    return schema[["source", "column_name", "column_type"]]


schema = pd.concat(
    [
        describe_relation(FACT_DAILY_MARCH, "fact_content_daily_performance/month=2026-03"),
        describe_relation(DIM_CLIENTS, "dim_clients"),
        describe_relation(DIM_CONTENT, "dim_content"),
    ],
    ignore_index=True,
)


def categorize_field(column_name: str) -> tuple[str, str]:
    if column_name in feature_source_fields:
        return "feature source", "Used only through lagged or trailing transformations."
    if column_name in context_fields or column_name.endswith("_hash_id"):
        return "context", "Join, filter, grouping, or audit field only."
    if column_name.endswith("_available"):
        return "context", "Availability flag used for filtering and data-quality checks."
    if column_name.startswith("ga4_") or column_name in {"sessions_ai", "scroll_events"}:
        return "excluded", "Engagement/AI fields are sparse and need a separate availability-aware contract."
    if column_name in {"month"}:
        return "context", "Partition or time-window control."
    return "excluded", "Not in this five-feature contract until separately reviewed."


if schema.empty:
    print("Schema categorization skipped until HF_TOKEN is available.")
    planned_contract = pd.DataFrame(
        {
            "bucket": ["feature"] * len(feature_fields) + ["label/proxy"] * len(label_fields) + ["context"] * len(context_fields),
            "field": feature_fields + label_fields + context_fields,
        }
    )
    display(planned_contract)
else:
    categorized = schema.copy()
    categorized[["bucket", "reason"]] = categorized["column_name"].apply(lambda name: pd.Series(categorize_field(name)))
    display(categorized.sort_values(["bucket", "source", "column_name"]))
    print("Discovered fields categorized:", len(categorized))

print("Feature fields:", feature_fields)
print("Label/proxy field:", label_fields)


,source,column_name,column_type,bucket,reason
35,dim_clients,access_profile,VARCHAR,context,"Join, filter, grouping, or audit field only."
31,dim_clients,client_hash_id,VARCHAR,context,"Join, filter, grouping, or audit field only."
39,dim_clients,ga4_data_start,DATE,context,"Join, filter, grouping, or audit field only."
38,dim_clients,gsc_data_start,DATE,context,"Join, filter, grouping, or audit field only."
40,dim_content,client_hash_id,VARCHAR,context,"Join, filter, grouping, or audit field only."
...,...,...,...,...,...
18,fact_content_daily_performance/month=2026-03,sessions_referral,BIGINT,excluded,Not in this five-feature contract until separa...
19,fact_content_daily_performance/month=2026-03,sessions_social,BIGINT,excluded,Not in this five-feature contract until separa...
10,fact_content_daily_performance/month=2026-03,gsc_avg_position,DOUBLE,feature source,Used only through lagged or trailing transform...
8,fact_content_daily_performance/month=2026-03,gsc_clicks,BIGINT,feature source,Used only through lagged or trailing transform...


Discovered fields categorized: 66
Feature fields: ['decision_dow', 'gsc_impressions_lag1', 'gsc_clicks_lag1', 'gsc_ctr_7d_past', 'gsc_position_7d_past_avg']
Label/proxy field: ['label_next_day_gsc_clicks']


## 3. Verify it with queries

The next three cells are the contract verification queries. They use only `month = '2026-03'` and do not touch the sealed final month.

### Cell A — Verify Grain

This query should return zero rows. Any returned row would mean the claimed grain (`report_date`, `client_hash_id`, `content_hash_id`) is duplicated.

In [4]:
verify_grain_query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM {FACT_DAILY_MARCH}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 10
"""

grain_check = run_sql(verify_grain_query, "Cell A: duplicate rows at claimed grain")
if EXECUTE_REMOTE_QUERIES:
    print("Grain holds:", grain_check.empty)



--- Cell A: duplicate rows at claimed grain ---
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 10


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


Grain holds: True


### Cell B — Row Count + Date Span

This query checks how many March rows exist, how many distinct report dates are present, and whether the date span stays inside the development month.

In [5]:
row_count_date_span_query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT report_date) AS distinct_days,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date,
    COUNT(DISTINCT client_hash_id) AS distinct_clients,
    COUNT(DISTINCT content_hash_id) AS distinct_content_items
FROM {FACT_DAILY_MARCH}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
"""

row_count_date_span = run_sql(row_count_date_span_query, "Cell B: March row count and date span")



--- Cell B: March row count and date span ---
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT report_date) AS distinct_days,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date,
    COUNT(DISTINCT client_hash_id) AS distinct_clients,
    COUNT(DISTINCT content_hash_id) AS distinct_content_items
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'


,total_rows,distinct_days,min_report_date,max_report_date,distinct_clients,distinct_content_items
0,9841378,31,2026-03-01,2026-03-31,55,331437


### Cell C — Availability with `IS TRUE`

This query proves why availability flags must be handled with three-valued logic. It separately counts `TRUE`, `FALSE`, and `NULL`, then reports how many rows survive a strict `ga4_data_available IS TRUE` filter.

In [6]:
availability_is_true_query = f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_true_rows,
    SUM(CASE WHEN ga4_data_available IS FALSE THEN 1 ELSE 0 END) AS ga4_available_false_rows,
    SUM(CASE WHEN ga4_data_available IS NULL THEN 1 ELSE 0 END) AS ga4_available_null_rows,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_surviving_is_true_filter
FROM {FACT_DAILY_MARCH}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
"""

availability_check = run_sql(availability_is_true_query, "Cell C: GA4 availability with IS TRUE / IS FALSE / IS NULL")



--- Cell C: GA4 availability with IS TRUE / IS FALSE / IS NULL ---
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_true_rows,
    SUM(CASE WHEN ga4_data_available IS FALSE THEN 1 ELSE 0 END) AS ga4_available_false_rows,
    SUM(CASE WHEN ga4_data_available IS NULL THEN 1 ELSE 0 END) AS ga4_available_null_rows,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_surviving_is_true_filter
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_true_rows,ga4_available_false_rows,ga4_available_null_rows,rows_surviving_is_true_filter
0,9841378,413966.0,6408671.0,3018741.0,413966.0


### Cell D — Five Features (max)

The feature frame below uses exactly five model features. Each feature is knowable at the decision moment because it is either calendar information or uses only prior rows in March. The label is next-day GSC clicks and is never included in the honest feature matrix.

In [7]:
feature_frame_query = f"""
WITH base AS (
    SELECT
        CAST(report_date AS DATE) AS report_date,
        client_hash_id,
        content_hash_id,
        COALESCE(gsc_impressions, 0) AS gsc_impressions,
        COALESCE(gsc_clicks, 0) AS gsc_clicks,
        gsc_avg_position
    FROM {FACT_DAILY_MARCH}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
),
windowed AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        EXTRACT(DOW FROM report_date) AS decision_dow,
        LAG(gsc_impressions, 1) OVER content_day AS gsc_impressions_lag1,
        LAG(gsc_clicks, 1) OVER content_day AS gsc_clicks_lag1,
        SUM(gsc_clicks) OVER trailing_7 / NULLIF(SUM(gsc_impressions) OVER trailing_7, 0) AS gsc_ctr_7d_past,
        AVG(gsc_avg_position) OVER trailing_7 AS gsc_position_7d_past_avg,
        LEAD(gsc_clicks, 1) OVER content_day AS label_next_day_gsc_clicks
    FROM base
    WINDOW
        content_day AS (PARTITION BY client_hash_id, content_hash_id ORDER BY report_date),
        trailing_7 AS (PARTITION BY client_hash_id, content_hash_id ORDER BY report_date ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING)
)
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    decision_dow,
    gsc_impressions_lag1,
    gsc_clicks_lag1,
    gsc_ctr_7d_past,
    gsc_position_7d_past_avg,
    label_next_day_gsc_clicks
FROM windowed
WHERE report_date >= DATE '2026-03-08'
  AND report_date < DATE '2026-03-31'
  AND gsc_impressions_lag1 IS NOT NULL
  AND gsc_clicks_lag1 IS NOT NULL
  AND gsc_ctr_7d_past IS NOT NULL
  AND gsc_position_7d_past_avg IS NOT NULL
  AND label_next_day_gsc_clicks IS NOT NULL
  AND gsc_impressions_lag1 >= 10
LIMIT 50000
"""

feature_frame = run_sql(feature_frame_query, "Cell D: five-feature March modeling frame")

if not feature_frame.empty:
    feature_frame = feature_frame.copy()
    feature_frame[feature_fields] = feature_frame[feature_fields].replace([np.inf, -np.inf], np.nan)
    feature_frame = feature_frame.dropna(subset=feature_fields + label_fields)
    print("Feature frame shape:", feature_frame.shape)
    print("Feature columns:", feature_fields)
    print("Label column:", label_fields[0])
    display(feature_frame.head())
else:
    print("Feature frame is empty until remote execution is enabled.")

# Knowable at the decision moment because report_date day-of-week is calendar information.
# Knowable at the decision moment because gsc_impressions_lag1 uses the prior content-day only.
# Knowable at the decision moment because gsc_clicks_lag1 uses the prior content-day only.
# Knowable at the decision moment because gsc_ctr_7d_past uses only the previous 7 content-days.
# Knowable at the decision moment because gsc_position_7d_past_avg uses only the previous 7 content-days.



--- Cell D: five-feature March modeling frame ---
WITH base AS (
    SELECT
        CAST(report_date AS DATE) AS report_date,
        client_hash_id,
        content_hash_id,
        COALESCE(gsc_impressions, 0) AS gsc_impressions,
        COALESCE(gsc_clicks, 0) AS gsc_clicks,
        gsc_avg_position
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
),
windowed AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        EXTRACT(DOW FROM report_date) AS decision_dow,
        LAG(gsc_impressions, 1) OVER content_day AS gsc_impressions_lag1,
        LAG(gsc_clicks, 1) OVER content_day AS gsc_clicks_lag1,
        SUM(gsc_clicks) OVER trailing_7 / NULLIF(SUM(gsc_impressions) OVER trailing_7, 0) AS gsc_ctr_7d_past,
        AVG(gsc_avg_position) OVER trailing_7 AS gsc_position_7d_past_avg,
        LEA

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,decision_dow,gsc_impressions_lag1,gsc_clicks_lag1,gsc_ctr_7d_past,gsc_position_7d_past_avg,label_next_day_gsc_clicks
0,2026-03-24,client_62f4a7e64f5e0096,content_0548a0e7e8fca67a,2,79,0,0.002358,4.991439,0
1,2026-03-25,client_62f4a7e64f5e0096,content_0548a0e7e8fca67a,3,67,0,0.000000,5.333181,0
2,2026-03-26,client_62f4a7e64f5e0096,content_0548a0e7e8fca67a,4,72,0,0.000000,4.335165,0
3,2026-03-27,client_62f4a7e64f5e0096,content_0548a0e7e8fca67a,5,54,0,0.000000,4.771673,1
4,2026-03-28,client_62f4a7e64f5e0096,content_0548a0e7e8fca67a,6,109,0,0.000000,3.949471,0
...,...,...,...,...,...,...,...,...,...
49995,2026-03-20,client_400c21c81c8b46ef,content_f73ad2bc456869e9,5,13,0,0.000000,6.745936,0
49996,2026-03-22,client_400c21c81c8b46ef,content_f73ad2bc456869e9,0,14,0,0.000000,6.502739,0
49997,2026-03-24,client_400c21c81c8b46ef,content_f73ad2bc456869e9,2,10,0,0.000000,6.229723,0
49998,2026-03-25,client_400c21c81c8b46ef,content_f73ad2bc456869e9,3,25,0,0.000000,6.232748,0


Feature frame shape: (50000, 9)
Feature columns: ['decision_dow', 'gsc_impressions_lag1', 'gsc_clicks_lag1', 'gsc_ctr_7d_past', 'gsc_position_7d_past_avg']
Label column: label_next_day_gsc_clicks


,report_date,client_hash_id,content_hash_id,decision_dow,gsc_impressions_lag1,gsc_clicks_lag1,gsc_ctr_7d_past,gsc_position_7d_past_avg,label_next_day_gsc_clicks
0,2026-03-24,client_62f4a7e64f5e0096,content_0548a0e7e8fca67a,2,79,0,0.002358,4.991439,0
1,2026-03-25,client_62f4a7e64f5e0096,content_0548a0e7e8fca67a,3,67,0,0.000000,5.333181,0
2,2026-03-26,client_62f4a7e64f5e0096,content_0548a0e7e8fca67a,4,72,0,0.000000,4.335165,0
3,2026-03-27,client_62f4a7e64f5e0096,content_0548a0e7e8fca67a,5,54,0,0.000000,4.771673,1
4,2026-03-28,client_62f4a7e64f5e0096,content_0548a0e7e8fca67a,6,109,0,0.000000,3.949471,0


### Cell E — THE TRAP (Deliberate Leakage Experiment)

This experiment intentionally adds a label-derived column after training the honest model. If the score jumps dramatically, the lesson is that leakage can make a model look strong while teaching it the answer.

In [8]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score


def mean_cv_r2(X: pd.DataFrame, y: pd.Series) -> tuple[float, float]:
    model = RandomForestRegressor(
        n_estimators=80,
        max_depth=10,
        random_state=42,
        n_jobs=-1,
    )
    cv = KFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=cv, scoring="r2")
    return float(scores.mean()), float(scores.std())


if feature_frame.empty or len(feature_frame) < 50:
    print("Skipping leakage experiment until the March feature frame is available with at least 50 rows.")
else:
    model_data = feature_frame.sample(n=min(len(feature_frame), 20000), random_state=42).copy()
    X_honest = model_data[feature_fields]
    y = model_data[label_fields[0]]

    honest_score, honest_std = mean_cv_r2(X_honest, y)
    print(f"Honest feature CV R^2: {honest_score:.4f} +/- {honest_std:.4f}")

    rng = np.random.default_rng(42)
    model_data["leak_label_times_noise"] = y + rng.normal(0, max(float(y.std()), 1.0) * 0.01, size=len(y))
    X_leaky = model_data[feature_fields + ["leak_label_times_noise"]]
    leaky_score, leaky_std = mean_cv_r2(X_leaky, y)
    print(f"Leaky feature CV R^2: {leaky_score:.4f} +/- {leaky_std:.4f}")
    print(f"Score jump from leak: {leaky_score - honest_score:.4f}")

    model_data = model_data.drop(columns=["leak_label_times_noise"])
    print(f"Leak removed. Honest score to report: {honest_score:.4f}")
    print("Lesson learned: a label-derived feature can create impressive metrics by leaking the answer, not by learning usable search signals.")


Honest feature CV R^2: 0.4604 +/- 0.0519
Leaky feature CV R^2: 0.9743 +/- 0.0322
Score jump from leak: 0.5139
Leak removed. Honest score to report: 0.4604
Lesson learned: a label-derived feature can create impressive metrics by leaking the answer, not by learning usable search signals.


## 4. Data Limits

**Primary named limitation: GA4 availability is not uniform in the March slice.** Engagement fields are not safe default features unless filtered with `IS TRUE` and audited by client, because `ga4_data_available` can be `TRUE`, `FALSE`, or `NULL`.

Additional limitations:

- The warehouse is an unbalanced panel: clients have different start dates and history depth, so global windows may compare mature clients with newer clients.
- The March development month is useful for iteration, but it is still only one calendar month and may not represent seasonality or later behavior.
- Daily observed metrics are observational. They can support prioritization, but they cannot prove that refreshing a page causes traffic recovery.

In [9]:
data_limit_query = f"""
SELECT
    f.client_hash_id,
    c.access_profile,
    c.gsc_data_start,
    c.ga4_data_start,
    COUNT(*) AS march_rows,
    SUM(CASE WHEN f.ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_true_rows,
    SUM(CASE WHEN f.ga4_data_available IS FALSE THEN 1 ELSE 0 END) AS ga4_false_rows,
    SUM(CASE WHEN f.ga4_data_available IS NULL THEN 1 ELSE 0 END) AS ga4_null_rows,
    ROUND(SUM(CASE WHEN f.ga4_data_available IS TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS ga4_true_pct
FROM {FACT_DAILY_MARCH} AS f
LEFT JOIN {DIM_CLIENTS} AS c
USING (client_hash_id)
WHERE f.report_date >= DATE '2026-03-01'
  AND f.report_date < DATE '2026-04-01'
GROUP BY 1, 2, 3, 4
ORDER BY ga4_true_pct ASC, march_rows DESC
LIMIT 20
"""

data_limit_check = run_sql(data_limit_query, "Section 4: GA4 availability differs by client in March")



--- Section 4: GA4 availability differs by client in March ---
SELECT
    f.client_hash_id,
    c.access_profile,
    c.gsc_data_start,
    c.ga4_data_start,
    COUNT(*) AS march_rows,
    SUM(CASE WHEN f.ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_true_rows,
    SUM(CASE WHEN f.ga4_data_available IS FALSE THEN 1 ELSE 0 END) AS ga4_false_rows,
    SUM(CASE WHEN f.ga4_data_available IS NULL THEN 1 ELSE 0 END) AS ga4_null_rows,
    ROUND(SUM(CASE WHEN f.ga4_data_available IS TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS ga4_true_pct
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet') AS f
LEFT JOIN read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet') AS c
USING (client_hash_id)
WHERE f.report_date >= DATE '2026-03-01'
  AND f.report_date < DATE '2026-04-01'
GROUP BY 1, 2, 3, 4
ORDER BY ga4_true_pct ASC, march_rows DESC
LIMIT 20


,client_hash_id,access_profile,gsc_data_start,ga4_data_start,march_rows,ga4_true_rows,ga4_false_rows,ga4_null_rows,ga4_true_pct
0,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19,988497,37.0,988460.0,0.0,0.00
1,client_08a6a72ff48e62c0,gsc_only,2025-09-24,NaT,851275,0.0,0.0,851275.0,0.00
2,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT,756660,0.0,0.0,756660.0,0.00
3,client_2b4306c3ed003f01,gsc_only,2026-02-19,NaT,375906,0.0,0.0,375906.0,0.00
4,client_19b89ee4fe3db6da,gsc_and_ga4,NaT,2026-01-09,213001,3.0,212998.0,0.0,0.00
5,client_400c21c81c8b46ef,gsc_and_ga4,2025-10-26,NaT,106808,0.0,106808.0,0.0,0.00
6,client_795153d5b7850ccf,gsc_only,2025-09-24,NaT,78709,0.0,0.0,78709.0,0.00
7,client_599043c0ff13edea,gsc_only,2025-09-27,NaT,22537,0.0,0.0,22537.0,0.00
8,client_8dbf3abdf07569e0,gsc_only,2025-07-29,NaT,17825,0.0,0.0,17825.0,0.00
9,client_8ae2bfb5aa1ffa1e,gsc_only,2025-07-28,NaT,11067,0.0,0.0,11067.0,0.00


## Self-check

Before submitting, I confirm each line honestly:

- [x] Every section above is filled with markdown reasoning and code-backed checks.
- [x] The unit of analysis is stated as `report_date + client_hash_id + content_hash_id` and verified with a duplicate-grain query.
- [x] All development queries use `month = '2026-03'` and do not use `2026-06`.
- [x] The feature frame uses exactly five features, each knowable at the decision moment.
- [x] The label/proxy is separated from the feature matrix, and the leakage trap demonstrates why this matters.
- [x] The `IS TRUE` availability check distinguishes `TRUE`, `FALSE`, and `NULL`.
- [x] No tokens or credentials are hardcoded in the notebook.
- [x] Run the remote queries in Colab or a local environment with `HF_TOKEN` before final submission.
- [x] Commit this completed notebook under `work/notebooks/` after reviewing the outputs.